# Mapping of the Antarctic Penguin Data 

This project is used to map and analyze the data set [Antarctic Penguin Tracking & Colony Data](https://www.kaggle.com/datasets/ibrahimqasimi/antarctic-penguin-tracking-and-colony-data?select=MAPPPD_Antarctic_Penguin_Colonies.csv).



In [37]:
# imports and setups
from pathlib import Path
import webbrowser

import folium
from folium.plugins import HeatMap
import geopandas as gpd
import numpy as np
import pandas as pd

Since VS-Code didn't output the folium Maps properly, I created a helper function to save the map as file and open it automatically in a browser window

In [ ]:
# helper function for saving maps
def save_map(geo_map: folium.Map, path: str) -> None:
    """
    Automatically saves the folium.Map as html file and opens it in the webbrowser.
    
    Args:
        geo_map: The created folium map
        path: The name of the file 
    """
    
    file_path = Path(path).resolve()
    geo_map.save(file_path)
    webbrowser.open(file_path.as_uri())

## Loading & Cleaning the Datasets

In [39]:
# storing the paths 
colonies_data = Path(r'data\MAPPPD_Antarctic_Penguin_Colonies.csv').resolve()
adelie_data = Path(r'data\NOAA_Adelie_Penguin_Telemetry_1997_2013.csv').resolve()
occurrences_data = Path(r'data\OBIS_Penguin_Occurrences_Global.csv').resolve()

# creating df's 
colonies = pd.read_csv(colonies_data)
adelies = pd.read_csv(adelie_data)
occurrences = pd.read_csv(occurrences_data)

#### Colonies

In [ ]:
# Penguin Colonies first entries
colonies.head()

,site_name,site_id,cammlr_region,longitude_epsg_4326,latitude_epsg_4326,common_name,day,month,year,season_starting,penguin_count,accuracy,count_type,vantage,reference
0,Acuna Island,ACUN,48.2,-44.637,-60.761,adelie penguin,NaN,NaN,1993,1993,2008.0,1.0,nests,ground,Coria N. R. D. Montalti E. F. Rombol&aacute...
1,Acuna Island,ACUN,48.2,-44.637,-60.761,adelie penguin,NaN,NaN,1994,1994,1920.0,1.0,nests,NaN,Woehler E. J. and J. P. Croxall (1997). &ldqu...
2,Acuna Island,ACUN,48.2,-44.637,-60.761,adelie penguin,NaN,NaN,2004,2004,1880.0,1.0,nests,ground,Coria N. R. D. Montalti E. F. Rombol&aacute...
3,Acuna Island,ACUN,48.2,-44.637,-60.761,adelie penguin,25.0,2.0,2011,2010,3079.0,5.0,nests,vhr,Lynch H. J. and M. A. LaRue (2014). &ldquo;Fi...
4,Acuna Island,ACUN,48.2,-44.637,-60.761,chinstrap penguin,28.0,12.0,1983,1983,4000.0,4.0,nests,ground,Poncet S. and J. Poncet (1985). &ldquo;A surv...


In [41]:
# Penguin colonies describe
colonies.describe()

,longitude_epsg_4326,latitude_epsg_4326,day,month,year,season_starting,penguin_count,accuracy
count,5407.000000,5407.000000,3305.000000,3902.000000,5407.000000,5407.000000,5389.000000,5389.000000
mean,-14.344096,-65.735702,15.356732,6.678114,2002.434992,2002.083595,7092.615142,1.716459
std,85.276329,4.423644,8.674753,5.252060,15.863941,15.782279,25932.391891,1.200775
min,-157.700000,-77.710000,1.000000,1.000000,1892.000000,1892.000000,0.000000,1.000000
25%,-62.673000,-66.674000,8.000000,1.000000,1990.000000,1990.000000,212.000000,1.000000
50%,-58.933000,-64.765000,15.000000,10.000000,2007.000000,2007.000000,1000.000000,1.000000
75%,-38.988500,-62.458000,23.000000,12.000000,2014.000000,2014.000000,3440.000000,2.000000
max,171.169000,-60.554000,31.000000,12.000000,2025.000000,2024.000000,504332.000000,5.000000


#### Adelie Penguins

In [ ]:
# Adelie Penguin Telemetry first entries
adelies.head()

,BirdId,Sex,Age,Breed Stage,DateGMT,TimeGMT,Latitude,Longitude,ArgosQuality
0,ADPE1,female,adult,incubation,28/10/1997,7:54:00,-62.171667,-58.445000,2
1,ADPE1,female,adult,incubation,28/10/1997,9:32:00,-62.173333,-58.463333,2
2,ADPE1,female,adult,incubation,28/10/1997,18:15:00,-62.158333,-58.426667,1
3,ADPE1,female,adult,incubation,28/10/1997,19:57:00,-62.175000,-58.441667,2
4,ADPE1,female,adult,incubation,28/10/1997,21:37:00,-62.171667,-58.445000,2


In [ ]:
# Adelie Penguin Telemetry describe
adelies.describe()

,Latitude,Longitude
count,12722.000000,12722.000000
mean,-63.265227,-55.509622
std,1.907643,4.306895
min,-70.765000,-65.860000
25%,-63.503000,-58.446000
50%,-62.334000,-58.227000
75%,-62.177000,-52.043250
max,-61.151000,-44.220000


#### Penguin Occurrences

In [ ]:
# Penguin Occurrences first entries
occurrences.head()

,species,scientific_name,latitude,longitude,date,year,month,day,country,locality,depth,basis_of_record,dataset_id,institution,collection,catalog_number
0,Adelie Penguin,Pygoscelis adeliae,-74.780998,169.535004,1.008115e+12,2001.0,12.0,12.0,Antarctica,Inexpressible Island,0.0,HumanObservation,8a1109fe-bf82-42b2-b296-a46c8059b82b,AADC,ARGOS,ARGOS-915-0531
1,Adelie Penguin,Pygoscelis adeliae,-74.337997,165.123993,7.875360e+11,1994.0,12.0,16.0,Antarctica,Edmonson Point,0.0,HumanObservation,8a1109fe-bf82-42b2-b296-a46c8059b82b,AADC,ARGOS,ARGOS-45-0030
2,Adelie Penguin,Pygoscelis adeliae,-67.357002,62.550999,1.074989e+12,2004.0,1.0,25.0,Antarctica,Bechervaise Island,0.0,HumanObservation,8a1109fe-bf82-42b2-b296-a46c8059b82b,AADC,ARGOS,ARGOS-1502-0330
3,Adelie Penguin,Pygoscelis adeliae,-66.448642,47.588606,8.952768e+11,1998.0,5.0,16.0,NaN,"Bechervaise Island, Mawson Coast",NaN,MachineObservation,48cb8624-a221-47ed-9a6d-b99b0bb394e0,NaN,NaN,NaN
4,Adelie Penguin,Pygoscelis adeliae,-74.630216,166.545896,8.494848e+11,1996.0,12.0,2.0,NaN,"Edmonson Point, Ross Sea",NaN,MachineObservation,48cb8624-a221-47ed-9a6d-b99b0bb394e0,NaN,NaN,NaN


In [ ]:
# Penguin Occurrences describe
occurrences.describe()

,latitude,longitude,date,year,month,day,depth
count,27047.000000,27047.000000,2.665200e+04,26652.000000,25323.000000,25228.000000,6059.000000
mean,-60.218206,22.783076,1.050349e+12,2002.911339,4.898906,15.651221,0.138967
std,6.823451,71.190096,3.179978e+11,10.053934,4.542827,9.115190,16.299509
min,-78.483300,-179.955314,-2.445725e+12,1892.000000,1.000000,1.000000,-70.000000
25%,-65.358961,-45.633000,9.165312e+11,1999.000000,1.000000,8.000000,0.000000
50%,-61.607500,47.296696,1.074989e+12,2004.000000,2.000000,16.000000,0.000000
75%,-53.470717,66.287351,1.233878e+12,2009.000000,11.000000,24.000000,0.000000
max,-24.049999,179.680610,1.762906e+12,2025.000000,12.000000,31.000000,1253.000000


## Creating a Map

First I want to map the colonies with a heatmap to see the density of penguin colonies

In [52]:
# defining the range for the map to show
c_map_location = [colonies['latitude_epsg_4326'].mean(), colonies['longitude_epsg_4326'].mean()]

# creating the map
c_map = folium.Map(location=c_map_location, tiles='Cartodb dark_matter', zoom_start=2)

# adding a heatmap
HeatMap(data=colonies[['latitude_epsg_4326', 'longitude_epsg_4326']],radius=10).add_to(c_map) 

# save & display in webbrowser
save_map(c_map, 'c_map.html')

In [50]:
# defining the range for map to show
o_map_locations = [occurrences['latitude'].mean(), occurrences['longitude'].mean()]

# creating the map
o_map = folium.Map(location=o_map_locations, tiles='Cartodb dark_matter', zoom_start=3)

# adding a heatmap
HeatMap(data=occurrences[['latitude', 'longitude']], radius=10).add_to(o_map)

# save & display in browser
save_map(o_map, 'o_map.html')